In [2]:
import pandas as pd
import xlrd as xl
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 
import numpy as np 

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
# Inladen van de dataset
df = pd.read_excel('AmesHousing.xlsx')

# One-hot encoding: verandert de tekst-categorie 'Neighborhood' in kolommen met 1 en 0
# drop_first=True voorkomt overlappende data
df_encoded = pd.get_dummies(df, columns=['Neighborhood'], drop_first=True)

In [ ]:
# Selecteren van de features (X) en het doel dat we willen voorspellen (y)
feature_cols = ['Overall Qual', 'Gr Liv Area'] + [col for col in df_encoded.columns if 'Neighborhood_' in col]
X = df_encoded[feature_cols]
y = df_encoded['SalePrice']

# Splitsen van de data: 80% om het model te trainen, 20% om het model te testen (examen)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialiseren van het lineaire regressiemodel met hyperparameters
model = LinearRegression(fit_intercept=True, n_jobs=-1)

# Het model de patronen laten leren uit de trainingsdata
model.fit(X_train, y_train)

# Voorspellingen maken op basis van de testdata die het model nog nooit heeft gezien
y_pred = model.predict(X_test)

# Berekenen van de foutmarge (MAE) en (R2)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Mean Squared Error (MSE): ${mse:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-kwadraat (R2): {r2:.4f}")

Mean Absolute Error (MAE): $24,736.74
Mean Squared Error (MSE): $1,525,093,881.39
Root Mean Squared Error (RMSE): $39,052.45
R-kwadraat (R2): 0.8098


: 

In [5]:
model = LinearRegression(fit_intercept=True, n_jobs=-1)

model.fit(X_train, y_train)

print("Het model is succesvol getraind!")

Het model is succesvol getraind!


In [6]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Evaluatiemetrieken Testset ---")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-kwadraat (R2): {r2:.4f}")

--- Evaluatiemetrieken Testset ---
Mean Absolute Error (MAE): $24,736.74
Root Mean Squared Error (RMSE): $39,052.45
R-kwadraat (R2): 0.8098


In [7]:
# Meer features toevoegen om het model meer context te geven
feature_cols_exp = ['Overall Qual', 'Gr Liv Area', 'Year Built', 'Total Bsmt SF'] + \
                   [col for col in df_encoded.columns if 'Neighborhood_' in col]

# Lege waarden in de kelder-kolom opvullen met 0 om errors te voorkomen
df_encoded['Total Bsmt SF'] = df_encoded['Total Bsmt SF'].fillna(0)

X_exp = df_encoded[feature_cols_exp]
y_exp = df_encoded['SalePrice']

# Opnieuw splitsen en het nieuwe model trainen
X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(X_exp, y_exp, test_size=0.2, random_state=42)
model_expA = LinearRegression()
model_expA.fit(X_train_exp, y_train_exp)
y_pred_expA = model_expA.predict(X_test_exp)

print("Resultaten Experiment A (Extra features):")
print(f"MAE: ${mean_absolute_error(y_test_exp, y_pred_expA):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp, y_pred_expA):.4f}")

Resultaten Experiment A (Extra features):
MAE: $22,532.72
R-kwadraat (R2): 0.8320


In [14]:
# StandardScaler drukt alle waardes samen zodat grote getallen (zoals oppervlakte) 
# niet onterecht zwaarder wegen dan kleine getallen (zoals aantal kamers).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_exp)
X_test_scaled = scaler.transform(X_test_exp)

# Baseline SGD: Optimalisatie in stapjes in plaats van een vaste wiskundige formule
sgd_baseline = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.001, random_state=42)
sgd_baseline.fit(X_train_scaled, y_train_exp)
y_pred_base = sgd_baseline.predict(X_test_scaled)

# Experiment SGD: Meer iteraties (epochs) en kleinere stapjes (eta0) voor meer precisie
sgd_exp = SGDRegressor(max_iter=5000, learning_rate='constant', eta0=0.0001, random_state=42)
sgd_exp.fit(X_train_scaled, y_train_exp)
y_pred_exp = sgd_exp.predict(X_test_scaled)

print("Resultaten SGD Experiment (Meer epochs, kleinere stapjes):")
print(f"MAE: ${mean_absolute_error(y_test_exp, y_pred_exp):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp, y_pred_exp):.4f}")

Resultaten SGD Experiment (Meer epochs, kleinere stapjes):
MAE: $22,560.53
R-kwadraat (R2): 0.8319


In [ ]:
# We encoden nu 'Neighborhood', 'House Style' en 'Garage'
df_encoded_exp3 = pd.get_dummies(df, columns=['Neighborhood', 'House Style', 'Garage'], drop_first=True)

# Vul eventuele missende waarden in met 0 voor kelders (Total Bsmt SF)
df_encoded_exp3['Total Bsmt SF'] = df_encoded_exp3['Total Bsmt SF'].fillna(0)

# Stel de nieuwe featurelijst samen met de kolommen die wél in jouw dataset staan
# Let op: we gebruiken nu 'Lot Area' en 'Full Bath' als extra numerieke features!
feature_cols_exp3 = ['Overall Qual', 'Gr Liv Area', 'Year Built', 'Total Bsmt SF', 'Lot Area', 'Full Bath'] + \
                   [col for col in df_encoded_exp3.columns if 'Neighborhood_' in col or 'House Style_' in col or 'Garage_' in col]

X_exp3 = df_encoded_exp3[feature_cols_exp3]
y_exp3 = df_encoded_exp3['SalePrice']

# Splits de data opnieuw
X_train_exp3, X_test_exp3, y_train_exp3, y_test_exp3 = train_test_split(
    X_exp3, y_exp3, test_size=0.2, random_state=42
)

# Train een standaard Linear Regression model op deze nieuwe dataset
model_exp3 = LinearRegression()
model_exp3.fit(X_train_exp3, y_train_exp3)
y_pred_exp3 = model_exp3.predict(X_test_exp3)

print("Resultaten Experiment 3 (Extra features: Lot Area, Full Bath, House Style & Garage)")
print(f"MAE: ${mean_absolute_error(y_test_exp3, y_pred_exp3):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp3, y_pred_exp3):.4f}")

--- Resultaten Experiment 3 (Extra features: Lot Area, Full Bath, House Style & Garage) ---
MAE: $21,881.90
R-kwadraat (R2): 0.8373


In [ ]:
# Experiment 4: SGDRegressor met 'Adaptive' Learning Rate aantonen (Divergentie)
scaler3 = StandardScaler()
X_train_scaled3 = scaler3.fit_transform(X_train_exp3)
X_test_scaled3 = scaler3.transform(X_test_exp3)

# De learning rate (eta0) staat hier te hoog (0.01) met instelling 'adaptive'
sgd_exp_adaptive = SGDRegressor(max_iter=10000, learning_rate='adaptive', eta0=0.01, random_state=42)
sgd_exp_adaptive.fit(X_train_scaled3, y_train_exp3)
y_pred_sgd_adapt = sgd_exp_adaptive.predict(X_test_scaled3)

# Dit laat exploding gradients zien: het model neemt te grote stappen en mist de oplossing volledig
print("Resultaten RUN 4")
print(f"MAE: ${mean_absolute_error(y_test_exp3, y_pred_sgd_adapt):,.2f}")
print(f"R-kwadraat (R2): {r2_score(y_test_exp3, y_pred_sgd_adapt):.4f}")

Resultaten RUN 4: SGD Divergentie
MAE: $414,989,577.80
R-kwadraat (R2): -52720729.7511
